In [7]:
from __future__ import annotations

from collections.abc import Iterator

import cirq
#import qiskit
#from qiskit.quantum_info import hellinger_fidelity



sys.path.insert(0, os.path.join(cd, 'supermarq-benchmarks/supermarq/benchmarks'))
sys.path.insert(0, os.path.join(cd, 'supermarq-benchmarks/supermarq'))
sys.path.insert(0, os.path.join(cd, 'supermarq-benchmarks'))

from supermarq.benchmark import Benchmark

class GHZ(Benchmark):
    """Represents the GHZ state preparation benchmark parameterized by the number of qubits n.

    Device performance is based on the Hellinger fidelity between the experimental and ideal
    probability distributions.
    """

    def __init__(self, num_qubits: int, method: str = "ladder") -> None:
        """Initialize a `GHZ` object.

        Args:
            num_qubits: Number of qubits in GHZ circuit.
            method: Circuit construction method to use. Must be "ladder", "star", or "logdepth". The
                "ladder" method uses a linear-depth CNOT ladder, appropriate for nearest-neighbor
                architectures. The "star" method is also linear depth, but with all CNOTs sharing
                the same control qubit. The "logdepth" method uses a log-depth CNOT fanout circuit.
        """
        if method not in ("ladder", "star", "logdepth"):
            raise ValueError(
                f"'{method}' is not a valid GHZ circuit construction method. Valid options are "
                "'ladder', 'star', and 'logdepth'."
            )
        self.num_qubits = num_qubits
        self.method = method

    

    # def qiskit_circuit(self) -> qiskit.QuantumCircuit:
    #     """Generate an n-qubit GHZ qiskit circuit.

    #     Returns:
    #         A `qiskit.QuantumCircuit`.
    #     """
    #     circuit = qiskit.QuantumCircuit(self.num_qubits, self.num_qubits)
    #     circuit.h(0)

    #     if self.method == "ladder":
    #         for i in range(1, self.num_qubits):
    #             circuit.cx(i - 1, i)

    #     elif self.method == "star":
    #         for i in range(1, self.num_qubits):
    #             circuit.cx(0, i)

    #     else:
    #         for i, j in _fanout(*range(self.num_qubits)):
    #             circuit.cx(i, j)

    #     for i in range(self.num_qubits):
    #         circuit.measure(i, i)

    #     return circuit

    # def score(self, counts: dict[str, float]) -> float:
    #     r"""Compute the Hellinger fidelity between the experimental and ideal results.

    #     The ideal results are 50% probabilty of measuring the all-zero state and 50% probability
    #     of measuring the all-one state.

    #     The formula for the Hellinger fidelity between two distributions p and q is given by
    #     $(\sum_i{p_i q_i})^2$.

    #     Args:
    #         counts: A dictionary containing the measurement counts from circuit execution.

    #     Returns:
    #         Hellinger fidelity as a float.
    #     """
    #     # Create an equal weighted distribution between the all-0 and all-1 states
    #     ideal_dist = {b * self.num_qubits: 0.5 for b in ["0", "1"]}
    #     total_shots = sum(counts.values())
    #     device_dist = {bitstr: count / total_shots for bitstr, count in counts.items()}
    #     return hellinger_fidelity(ideal_dist, device_dist)


def _fanout(*qubit_indices: int) -> Iterator[tuple[int, int]]:
    if len(qubit_indices) >= 2:
        cutoff = len(qubit_indices) // 2
        yield qubit_indices[0], qubit_indices[cutoff]
        yield from _fanout(*qubit_indices[:cutoff])
        yield from _fanout(*qubit_indices[cutoff:])


ModuleNotFoundError: No module named 'qiskit'

In [15]:
import cirq,qsimcirq
def circuit(num_qubits,method):
    """Generate an n-qubit GHZ cirq circuit.

    Returns:
    A `cirq.Circuit`.
    """
    qubits = cirq.LineQubit.range(num_qubits)
    circuit = cirq.Circuit()
    circuit += cirq.H(qubits[0])

    if method == "ladder":
        for i in range(1, num_qubits):
            circuit += cirq.CNOT(qubits[i - 1], qubits[i])

    elif method == "star":
        for i in range(1, num_qubits):
            circuit += cirq.CNOT(qubits[0], qubits[i])

    else:
        for i, j in _fanout(*range(num_qubits)):
            circuit += cirq.CNOT(qubits[i], qubits[j])

    circuit += cirq.measure(*qubits)
    return circuit

qc = circuit(31,"ladder")

In [16]:
qsim_options = {"t": 32}               

# Initialize the simulator with options
simulator = qsimcirq.QSimSimulator(qsim_options=qsim_options)

result = simulator.run(qc, repetitions=1000)

print("Measurement Results:")
print(result)

Measurement Results:
q(0),q(1),q(2),q(3),q(4),q(5),q(6),q(7),q(8),q(9),q(10),q(11),q(12),q(13),q(14),q(15),q(16),q(17),q(18),q(19),q(20),q(21),q(22),q(23),q(24),q(25),q(26),q(27),q(28),q(29),q(30)=...
